In [1]:
from pyspark.sql import SparkSession
from delta import *
from pyspark.sql.functions import avg, col, count

# 1️ Configuração da SparkSession com Delta Lake e Hive
warehouse_location = 'hdfs://hdfs-nn:9000/demo/Delta'

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Python Spark CSV (titles)") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(" SparkSession com Delta Lake iniciada.\n")

# 2️ Leitura do ficheiro CSV
csv_file_path = "titles.csv"

movies_df = spark.read.csv(
    csv_file_path,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

print(" Leitura do CSV concluída.\n")

# 3️ Análise do DataFrame
print("Schema do DataFrame 'titles.csv':")
movies_df.printSchema()

print("\n Alguns dados do DataFrame:")
movies_df.show(5, truncate=False)

print("\n Estatísticas descritivas (para colunas numéricas):")
movies_df.describe(['release_year', 'runtime']).show()

# 4️ Agrupar por 'type' e calcular média e contagem
print("\n A processar: Número de títulos e rating médio IMDb por tipo (excluindo short films)")

filtered_titles = movies_df.filter(
    (col("imdb_score").isNotNull()) &   
    (col("type") != "shortFilm")
)

grouped_titles = filtered_titles \
    .groupBy("type") \
    .agg(
        count("*").alias("num_titles"),
        avg("imdb_score").alias("avg_imdb_score")
    ) \
    .orderBy(col("num_titles").desc())

grouped_titles.show(truncate=False)

# 5️ Executar consulta SQL (Top 10 IMDb)
print("\n A executar consulta SQL sobre os dados do CSV...")

movies_df.createOrReplaceTempView("titles")

sql_query = """
    SELECT title, type, imdb_score
    FROM titles
    WHERE imdb_score IS NOT NULL
    ORDER BY imdb_score DESC
    LIMIT 10
"""

sql_df = spark.sql(sql_query)
sql_df.show(truncate=False)


 SparkSession com Delta Lake iniciada.

 Leitura do CSV concluída.

Schema do DataFrame 'titles.csv':
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: double (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)


 Alguns dados do DataFrame:
+--------+-----------------------------------+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------